[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01/blob/main/defect_detection.ipynb)

TODO: fix the button

# PBF Defect Detection

This notebook covers data loading, training, validation, and inference for detecting defects in Powder Bed Fusion images using a fine-tuned CNN.

## Clone GithHub repo

In [ ]:
!rm -rf mla_project/

In [1]:
import os

if not os.path.exists("/content/mla-prj-23-project-am04_group-am01") and not os.path.exists("/content/mla_project"):
  # DON'T SHARE THE PERSONAL ACCESS TOKEN

  # change the name of the branch here as needed
  !git clone -b vae https://***REMOVED-GITHUB-TOKEN***@github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01.git

  # Rename folder for simplicity
  !mv /content/mla-prj-23-project-am04_group-am01 /content/mla_project

!cd /content/mla_project && git pull

Cloning into 'mla-prj-23-project-am04_group-am01'...
remote: Enumerating objects: 3489, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 3489 (delta 7), reused 30 (delta 3), pack-reused 3450 (from 4)
Receiving objects: 100% (3489/3489), 1.95 GiB | 29.48 MiB/s, done.
Resolving deltas: 100% (1469/1469), done.
Already up to date.


## Install Dependencies

In [2]:
!pip install torch torchvision matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

## Imports

In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Original Dataset

## Dataset mean and std

In [ ]:
!python /content/mla_project/src/data_loader.py --data-dir /content/mla_project/images/original --compute-stats

Dataset mean (grayscale): 0.5839
Dataset std (grayscale): 0.2074


## Training - no augmentation

**IMPORTANT:**

- To perform K-Fold cross-validation, set --is_kfold to "True" and specify the number of folds with --k-folds. Example: --is_kfold "True", --k-folds 5

- To perform a single train/val split, set --is_kfold to "False" and specify the validation split ratio with --val-split. Example: --is_kfold "False", --val-split 0.2

- In both cases, to perform also testing, set --test to "True" and specify the test split ratio with --test-split. Example: --test "True", --test-split 0.2

### Define paths and parameters

In [ ]:
data_dir = '/content/mla_project/images/original'

# Training params
batch_size = 2
epochs = 10
learning_rate = 1e-5
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


### Launch training

In [ ]:
# k-fold cross validation (with test)

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2

### Plot Training & Validation Curves

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()


## Training - basic augmentations (simple transformations)

In [ ]:
data_dir = '/content/mla_project/images/original'

# Training params
batch_size = 2
epochs = 10
learning_rate = 1e-5
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# k-fold cross validation (with test) with augmented data

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2 \
    --aug "True"

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Variational Auto Encoders

## Training CVAE

In [15]:
!python /content/mla_project/external/Pytorch-VAE/train_cvae.py \
        --data_dir "/content/mla_project/images/original" \
        --batch_size 4 \
        --max_epoch 700 \
        --latent_size 128 \
        --image_size 512 \
        --device "cuda" \
        --kld_weight 0.5

Train dataset size:  64
Test dataset size:  16
Model created.

 Starting training...
Epoch: 0/700 Train loss: 724676.375, Train KLD: 324145.96875, Train Reconstruction Loss: 400530.53125
Epoch: 0/700 Test loss: 184943.765625, Test KLD: 213.20669555664062, Test Reconstruction Loss: 184943.765625
Saving model...
Epoch: 1/700 Train loss: 181432.796875, Train KLD: 49.42577362060547, Train Reconstruction Loss: 181383.34375
Epoch: 1/700 Test loss: 172895.03125, Test KLD: 13.763738632202148, Test Reconstruction Loss: 172895.03125
Epoch: 2/700 Train loss: 169624.828125, Train KLD: 12.363775253295898, Train Reconstruction Loss: 169612.46875
Epoch: 2/700 Test loss: 161500.84375, Test KLD: 6.452861785888672, Test Reconstruction Loss: 161500.84375
Epoch: 3/700 Train loss: 158487.640625, Train KLD: 4.874617099761963, Train Reconstruction Loss: 158482.75
Epoch: 3/700 Test loss: 150987.203125, Test KLD: 5.950540542602539, Test Reconstruction Loss: 150987.203125
Epoch: 4/700 Train loss: 148166.140625,

In [16]:
# Generate images

!python /content/mla_project/external/Pytorch-VAE/generate.py \
      --checkpoint ./checkpoints/model_699.pt \
      --latent_size 128 \
      --image_size 512 \
      --num_images 30 \
      --device cuda \
      --output_dir generated_images
      # model_type è di default cvae quindi non c'è bisogno di passarlo come parametro

Using CVAE model.
Model loaded from ./checkpoints/model_699.pt
Generated images saved to generated_images


In [17]:
!zip -r /content/generated_images.zip /content/generated_images

  adding: content/generated_images/ (stored 0%)
  adding: content/generated_images/image_27_class_1.png (deflated 2%)
  adding: content/generated_images/image_3_class_1.png (deflated 1%)
  adding: content/generated_images/image_5_class_1.png (deflated 1%)
  adding: content/generated_images/image_9_class_1.png (deflated 1%)
  adding: content/generated_images/image_7_class_1.png (deflated 1%)
  adding: content/generated_images/image_19_class_1.png (deflated 1%)
  adding: content/generated_images/image_2_class_0.png (deflated 1%)
  adding: content/generated_images/image_1_class_1.png (deflated 1%)
  adding: content/generated_images/image_23_class_1.png (deflated 1%)
  adding: content/generated_images/image_8_class_0.png (deflated 1%)
  adding: content/generated_images/image_29_class_1.png (deflated 1%)
  adding: content/generated_images/image_13_class_1.png (deflated 1%)
  adding: content/generated_images/image_17_class_1.png (deflated 1%)
  adding: content/generated_images/image_20_class

Push model to Huggingface

In [18]:
from huggingface_hub import login, create_repo, upload_file

# Effettua il login (ti verrà richiesto di inserire il token)
login()

# Carica un file nel repository
upload_file(
    path_or_fileobj="/content/checkpoints/model_699.pt",    # CHANGE FILE NAME WHEN NEEDED
    path_in_repo="Experiment11_model_699.pth",   #   CHANGE FILE NAME WHEN NEEDED
    repo_id="MLinAppl/cvae",
    repo_type="model"
)

model_699.pt:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/MLinAppl/cvae/commit/8f02931120c1c7ae0b7e60829648a2e74298a9b1', commit_message='Upload Experiment11_model_699.pth with huggingface_hub', commit_description='', oid='8f02931120c1c7ae0b7e60829648a2e74298a9b1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MLinAppl/cvae', endpoint='https://huggingface.co', repo_type='model', repo_id='MLinAppl/cvae'), pr_revision=None, pr_num=None)

In [19]:
#remove file to avoid confusion
!rm -rf generated_images/
!rm -rf generated_images.zip
!rm -rf checkpoints/
!rm -rf cvae_images/
!rm -rf checkpoints/

## Experiments CVAE

Upload model from HuggingFace and generate other images in order to do more experiments and calculate metrics.

In [36]:
from huggingface_hub import hf_hub_download
import shutil

# Download the file from the Hugging Face Hub
file_path = hf_hub_download(
    repo_id="MLinAppl/cvae",  # Your repository ID
    filename="Experiment11_model_699.pth",  # Name of the file in the repo
    repo_type="model"  # Specify the repo type
)

# Optional: copy it somewhere more accessible, like /content
destination_path = "/content/Experiment11_cvae_model_699.pth"
shutil.copy(file_path, destination_path)

print(f"File copied to: {destination_path}")


Experiment11_model_699.pth:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

File copied to: /content/Experiment11_cvae_model_699.pth


In [37]:
# Generate images

!python /content/mla_project/external/Pytorch-VAE/generate.py \
      --checkpoint /content/Experiment11_cvae_model_699.pth \
      --latent_size 128 \
      --image_size 512 \
      --num_images 30 \
      --device cuda \
      --output_dir additional_generated_images_cvae_exp

Using CVAE model.
Model loaded from /content/Experiment11_cvae_model_699.pth
Generated images saved to additional_generated_images_cvae_exp


In [38]:
!zip -r /content/additional_generated_images_cvae_exp.zip /content/additional_generated_images_cvae_exp

  adding: content/additional_generated_images_cvae_exp/ (stored 0%)
  adding: content/additional_generated_images_cvae_exp/image_27_class_1.png (deflated 1%)
  adding: content/additional_generated_images_cvae_exp/image_3_class_1.png (deflated 1%)
  adding: content/additional_generated_images_cvae_exp/image_5_class_1.png (deflated 2%)
  adding: content/additional_generated_images_cvae_exp/image_9_class_1.png (deflated 1%)
  adding: content/additional_generated_images_cvae_exp/image_7_class_1.png (deflated 2%)
  adding: content/additional_generated_images_cvae_exp/image_19_class_1.png (deflated 1%)
  adding: content/additional_generated_images_cvae_exp/image_2_class_0.png (deflated 1%)
  adding: content/additional_generated_images_cvae_exp/image_1_class_1.png (deflated 1%)
  adding: content/additional_generated_images_cvae_exp/image_23_class_1.png (deflated 1%)
  adding: content/additional_generated_images_cvae_exp/image_8_class_0.png (deflated 1%)
  adding: content/additional_generated_

In [35]:
#remove file to avoid confusion
!rm -rf additional_generated_images_cvae_exp.zip
!rm -rf additional_generated_images_cvae_exp/

## Training VAE

In [ ]:
!python /content/mla_project/external/Pytorch-VAE/train_vae.py \
        --data_dir "/content/mla_project/images/original" \
        --batch_size 4 \
        --max_epoch 200 \
        --latent_size 32 \
        --image_size 512 \
        --device "cuda" \
        #--load_epoch -1

In [ ]:
# Generate images

!python /content/mla_project/external/Pytorch-VAE/generate.py \
      --checkpoint ./checkpoints/model_199.pt \
      --latent_size 32 \
      --image_size 512 \
      --num_images 10 \
      --device cuda \
      --output_dir generated_images \
      --model_type "vae"

In [ ]:
!zip -r /content/generated_images.zip /content/generated_images

In [ ]:
from huggingface_hub import login, create_repo, upload_file

# Effettua il login (ti verrà richiesto di inserire il token)
login()

# Carica un file nel repository
upload_file(
    path_or_fileobj="/content/checkpoints/model_199.pt",    # CHANGE FILE NAME WHEN NEEDED
    path_in_repo="Experiment3_model_199.pth",   #   CHANGE FILE NAME WHEN NEEDED
    repo_id="MLinAppl/vae",
    repo_type="model"
)

In [ ]:
#remove file to avoid confusion
!rm -rf generated_images/
!rm -rf generated_images.zip
!rm -rf checkpoints/
!rm -rf images/
!rm -rf checkpoints/